In [4]:
from datasets import load_dataset
from collections import Counter
import gc
import ast
import pandas as pd

In [5]:
cleaned_df = pd.read_parquet("data/danbooru2025_cleaned.parquet")

In [6]:
# ---------------------------------
# Step 1: Filter
# ---------------------------------

lite_df = cleaned_df[
    (cleaned_df["score"] > 50)
].copy()
lite_df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url
52,9158186,2025-04-15T22:34:58.556-04:00,57,q,3565753,1girl arm_behind_head bed bed_sheet blush book...,,,https://cdn.donmai.us/360x360/d3/ba/d3ba038929...
79,9157865,2025-04-15T21:25:11.892-04:00,60,q,6399307,1girl animal_ear_fluff animal_ears ass back bl...,shinano_(azur_lane),azur_lane,https://cdn.donmai.us/360x360/b8/fa/b8fad959c4...
108,9157450,2025-04-15T19:44:02.219-04:00,61,s,8804416,1girl animal_ears ass bare_legs barefoot blue_...,shinano_(azur_lane) shinano_(visions_of_fantas...,azur_lane,https://cdn.donmai.us/360x360/21/81/218187464b...
115,9157370,2025-04-15T19:21:27.992-04:00,99,q,213816,1girl ahoge alternate_costume ass bikini black...,taihou_(azur_lane),azur_lane,https://cdn.donmai.us/360x360/ed/df/eddf7591e7...
117,9157353,2025-04-15T19:18:20.320-04:00,144,e,1103283,1boy 2girls 2koma ? black_nails black_pantyhos...,belle_(zenless_zone_zero) vivian_(zenless_zone...,zenless_zone_zero,https://cdn.donmai.us/360x360/3f/c4/3fc400f790...
...,...,...,...,...,...,...,...,...,...
4660208,29,2005-05-24T00:55:56.000-04:00,93,s,138002,1girl :d blonde_hair blush cowboy_shot crystal...,flandre_scarlet,touhou,https://cdn.donmai.us/360x360/b7/31/b7314742c0...
4660216,12,2005-05-24T00:25:31.000-04:00,74,e,84045,1boy 1girl :o anal_beads arms_behind_back bare...,maiko-chan_(goods_collector_maiko-chan),goods_collector_maiko-chan,https://cdn.donmai.us/360x360/19/a1/19a11d7311...
4660220,5,2005-05-23T23:57:53.000-04:00,83,q,62853,3girls :d ass bamboo_fence barefoot bath bathi...,kojima_kirie koyomi_hare_nanaka miharu_sena_ka...,girls_bravo,https://cdn.donmai.us/360x360/4f/5a/4f5ac0caa6...
4660222,3,2005-05-23T23:38:05.000-04:00,53,g,190709,1girl 2005 ahoge armor armored_dress blonde_ha...,artoria_pendragon_(fate) saber_(fate),fate/stay_night fate_(series),https://cdn.donmai.us/360x360/fd/b4/fdb47f79fb...


In [7]:
# ---------------------------------
# Step 2: Build combined tags
# ---------------------------------

def combine_tags(row):
    tags = []

    for col in [
        "tag_string_general",
        "tag_string_character",
        "tag_string_copyright",
    ]:
        value = row[col]
        if pd.notna(value) and value:
            tags.extend(value.split())

    return tags

lite_df["tags"] = lite_df.apply(combine_tags, axis=1)

In [20]:
counter = Counter()
for tags in lite_df["tags"]:
    counter.update(tags)

threshold = 100

keep_tags = {tag for tag, count in counter.items() if count >= threshold}

print(f"Threshold: {threshold}")
print(f"Original tags: {len(counter):,}")
print(f"Kept tags: {len(keep_tags):,}")
print(f"Removed tags: {len(counter) - len(keep_tags):,}")
print(f"Removed: {(1 - len(keep_tags)/len(counter)):.2%}")

num_images = sum(
    any(tag in keep_tags for tag in tags)
    for tags in lite_df["tags"]
)

print(f"Images with ≥1 kept tag: {num_images:,} / {len(lite_df):,}")
print(f"Percentage: {num_images / len(lite_df):.2%}")

orig_assignments = sum(len(tags) for tags in lite_df["tags"])
kept_assignments = sum(
    sum(tag in keep_tags for tag in tags)
    for tags in lite_df["tags"]
)

print(f"Original label assignments: {orig_assignments:,}")
print(f"Remaining label assignments: {kept_assignments:,}")
print(f"Removed assignments: {(1 - kept_assignments/orig_assignments):.2%}")

Threshold: 100
Original tags: 107,665
Kept tags: 11,516
Removed tags: 96,149
Removed: 89.30%
Images with ≥1 kept tag: 1,109,130 / 1,109,130
Percentage: 100.00%
Original label assignments: 44,929,923
Remaining label assignments: 43,992,748
Removed assignments: 2.09%


In [21]:
# ---------------------------------
# Keep images containing
# at least one top-50 tag
# ---------------------------------
lite_df["tags"] = lite_df["tags"].apply(
    lambda tags: [tag for tag in tags if tag in keep_tags]
)

lite_df = lite_df[lite_df["tags"].str.len() >= 2].reset_index(drop=True)
lite_df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url,tags
0,9158186,2025-04-15T22:34:58.556-04:00,57,q,3565753,1girl arm_behind_head bed bed_sheet blush book...,,,https://cdn.donmai.us/360x360/d3/ba/d3ba038929...,"[1girl, arm_behind_head, bed, bed_sheet, blush..."
1,9157865,2025-04-15T21:25:11.892-04:00,60,q,6399307,1girl animal_ear_fluff animal_ears ass back bl...,shinano_(azur_lane),azur_lane,https://cdn.donmai.us/360x360/b8/fa/b8fad959c4...,"[1girl, animal_ear_fluff, animal_ears, ass, ba..."
2,9157450,2025-04-15T19:44:02.219-04:00,61,s,8804416,1girl animal_ears ass bare_legs barefoot blue_...,shinano_(azur_lane) shinano_(visions_of_fantas...,azur_lane,https://cdn.donmai.us/360x360/21/81/218187464b...,"[1girl, animal_ears, ass, bare_legs, barefoot,..."
3,9157370,2025-04-15T19:21:27.992-04:00,99,q,213816,1girl ahoge alternate_costume ass bikini black...,taihou_(azur_lane),azur_lane,https://cdn.donmai.us/360x360/ed/df/eddf7591e7...,"[1girl, ahoge, alternate_costume, ass, bikini,..."
4,9157353,2025-04-15T19:18:20.320-04:00,144,e,1103283,1boy 2girls 2koma ? black_nails black_pantyhos...,belle_(zenless_zone_zero) vivian_(zenless_zone...,zenless_zone_zero,https://cdn.donmai.us/360x360/3f/c4/3fc400f790...,"[1boy, 2girls, 2koma, ?, black_nails, black_pa..."
...,...,...,...,...,...,...,...,...,...,...
1109125,29,2005-05-24T00:55:56.000-04:00,93,s,138002,1girl :d blonde_hair blush cowboy_shot crystal...,flandre_scarlet,touhou,https://cdn.donmai.us/360x360/b7/31/b7314742c0...,"[1girl, :d, blonde_hair, blush, cowboy_shot, c..."
1109126,12,2005-05-24T00:25:31.000-04:00,74,e,84045,1boy 1girl :o anal_beads arms_behind_back bare...,maiko-chan_(goods_collector_maiko-chan),goods_collector_maiko-chan,https://cdn.donmai.us/360x360/19/a1/19a11d7311...,"[1boy, 1girl, :o, anal_beads, arms_behind_back..."
1109127,5,2005-05-23T23:57:53.000-04:00,83,q,62853,3girls :d ass bamboo_fence barefoot bath bathi...,kojima_kirie koyomi_hare_nanaka miharu_sena_ka...,girls_bravo,https://cdn.donmai.us/360x360/4f/5a/4f5ac0caa6...,"[3girls, :d, ass, barefoot, bath, bathing, blu..."
1109128,3,2005-05-23T23:38:05.000-04:00,53,g,190709,1girl 2005 ahoge armor armored_dress blonde_ha...,artoria_pendragon_(fate) saber_(fate),fate/stay_night fate_(series),https://cdn.donmai.us/360x360/fd/b4/fdb47f79fb...,"[1girl, ahoge, armor, armored_dress, blonde_ha..."


In [28]:
# ---------------------------------
# Random sample
# ---------------------------------
lite_df = lite_df.sample(
    frac=1,
    random_state=42,
).reset_index(drop=True)

# ---------------------------------
# Train / Validation split
# ---------------------------------

val_split = 100_000
train_df = lite_df.iloc[:-val_split].reset_index(drop=True)
val_df = lite_df.iloc[-val_split:].reset_index(drop=True)

print(len(train_df), len(val_df))

1009130 100000


In [30]:
# Count tag frequencies
train_counter = Counter()
val_counter = Counter()

for tags in train_df["tags"]:
    train_counter.update(tags)

for tags in val_df["tags"]:
    val_counter.update(tags)

# Tags missing from validation
missing_in_val = set(train_counter) - set(val_counter)

print(f"Train tags: {len(train_counter):,}")
print(f"Val tags:   {len(val_counter):,}")
print(f"Missing in val: {len(missing_in_val):,}")

if missing_in_val:
    print("\nFirst 20 missing tags:")
    print(sorted(missing_in_val)[:20])


def print_stats(name, counter):
    min_tag, min_count = min(counter.items(), key=lambda x: x[1])

    print(f"\n{name} statistics")
    print("-" * (len(name) + 11))

    print(f"Rarest tag:")
    print(f"{min_tag}: {min_count} images")

    freqs = sorted(counter.values())

    print("\nTag frequency percentiles:")
    for p in [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]:
        idx = int((len(freqs) - 1) * p / 100)
        print(f"{p:>3}%: {freqs[idx]}")


print_stats("Training", train_counter)
print_stats("Validation", val_counter)

Train tags: 11,516
Val tags:   11,516
Missing in val: 0

Training statistics
-------------------
Rarest tag:
couter: 83 images

Tag frequency percentiles:
  0%: 83
  1%: 92
  5%: 101
 10%: 112
 25%: 160
 50%: 336
 75%: 1085
 90%: 4158
 95%: 10417
 99%: 66313
100%: 850817

Validation statistics
---------------------
Rarest tag:
frieren_stuck_in_a_mimic_(meme): 2 images

Tag frequency percentiles:
  0%: 2
  1%: 6
  5%: 8
 10%: 10
 25%: 16
 50%: 34
 75%: 108
 90%: 411
 95%: 1023
 99%: 6608
100%: 84075


In [31]:
!mkdir -p data

In [32]:
train_df.to_parquet(
    "data/danbooru2025_train.parquet",
    index=False,
    compression="zstd",
)
val_df.to_parquet(
    "data/danbooru2025_val.parquet",
    index=False,
    compression="zstd",
)